# Extracting Information From RAD/PACT Guides

The following code extracts housing data found in a PDF and formats it into tabular data. The [PDF](https://www.nyc.gov/assets/nycha/downloads/pdf/radpact-guide.pdf) is the New York City Housing Authority's latest list of converted PACT developments, made available in July 2025.

In [1]:
## import libraries
import pandas as pd
import numpy as np
import pdfplumber
import re
import csv

In [2]:
## read in the pdf
pdf = pdfplumber.open('../input/radpact-guide-07-2025.pdf')

In [3]:
## create empty lists for later
developments = []
extra_developments = []

In [4]:
## loop through pages and extract data
for page_num, page in enumerate(pdf.pages): # <-- for each page in the document...

    tables = page.extract_tables() # <-- exract every table
    print(f"page {page_num}: found {len(tables)} tables")

    for table in tables: # <-- then, for each table found...
        
        if table and re.match(r'^BBL$',table[0][0]): # <-- if it's a table and the first item in the first row in that table is BBL...
            headers = table[0] # <-- then first row is headers
            data = table[1:] # <-- remaining rows are data
            df = pd.DataFrame(columns=headers, data=data) # <-- transform into dataframe
            developments.append(df) # <-- append each dataframe (table) to an empty list
        else: # otherwise...
            extra_headers = ['BBL', 'BLDG#', 'M', 'ADDRESS', 'ZIP CODE', 'CD #', 'FC #', 'SS #', 'SA #', 'CC #', 'BIN#']
            extra = table
            dfs = pd.DataFrame(columns=headers, data=extra)
            extra_developments.append(dfs)

page 0: found 6 tables
page 1: found 2 tables
page 2: found 4 tables
page 3: found 2 tables
page 4: found 6 tables
page 5: found 5 tables
page 6: found 6 tables
page 7: found 2 tables
page 8: found 3 tables
page 9: found 3 tables
page 10: found 5 tables
page 11: found 2 tables
page 12: found 2 tables
page 13: found 6 tables
page 14: found 4 tables
page 15: found 4 tables
page 16: found 4 tables
page 17: found 2 tables
page 18: found 1 tables
page 19: found 1 tables
page 20: found 1 tables
page 21: found 3 tables
page 22: found 5 tables
page 23: found 5 tables
page 24: found 5 tables
page 25: found 5 tables
page 26: found 3 tables
page 27: found 5 tables
page 28: found 4 tables
page 29: found 4 tables
page 30: found 4 tables
page 31: found 6 tables
page 32: found 4 tables
page 33: found 2 tables
page 34: found 1 tables
page 35: found 1 tables
page 36: found 2 tables


In [5]:
## concat the dfs together into singular dfs
extras_combined = pd.concat(extra_developments, ignore_index=True)
combined_df = pd.concat(developments, ignore_index=True)

In [6]:
## now combine the two larger dfs into one
all_combined = pd.concat([extras_combined,combined_df], ignore_index=True)

In [7]:
## check number of addresses
all_combined.ADDRESS.nunique()

1538

In [8]:
all_combined.ADDRESS.unique().tolist()

['346 CLIFTON PLACE',
 '344 CLIFTON PLACE',
 '360 NOSTRAND AVENUE',
 '260 LEXINGTON AVENUE',
 '262 LEXINGTON AVENUE',
 '264 LEXINGTON AVENUE',
 '254 LEXINGTON AVENUE',
 '252 LEXINGTON AVENUE',
 '252A LEXINGTON AVENUE',
 '250 LEXINGTON AVENUE',
 '366 CLIFTON PLACE',
 '380 CLIFTON PLACE',
 '388 CLIFTON PLACE',
 '396 CLIFTON PLACE',
 '545 GREENE AVENUE',
 '555 GREENE AVENUE',
 '304 LEXINGTON AVENUE',
 '302 LEXINGTON AVENUE',
 '300 LEXINGTON AVENUE',
 '298 LEXINGTON AVENUE',
 '296 LEXINGTON AVENUE',
 '294 LEXINGTON AVENUE',
 '292 LEXINGTON AVENUE',
 '290 LEXINGTON AVENUE',
 '288 LEXINGTON AVENUE',
 '286 LEXINGTON AVENUE',
 '284 LEXINGTON AVENUE',
 '282 LEXINGTON AVENUE',
 '280 LEXINGTON AVENUE',
 '278 LEXINGTON AVENUE',
 '310 LEXINGTON AVENUE',
 '320 LEXINGTON AVENUE',
 '330 LEXINGTON AVENUE',
 '336 LEXINGTON AVENUE',
 '435 GATES AVENUE',
 '441 GATES AVENUE',
 '447 GATES AVENUE',
 '449 GATES AVENUE',
 '451 GATES AVENUE',
 '453 GATES AVENUE',
 '455 GATES AVENUE',
 '457 GATES AVENUE',
 '459 

<b>NOTE:</b> There's one BIN that's actually a blank, so the actual total number of unique BINs is 714. 

In [9]:
## check number of BINs
all_combined['BIN#'].nunique()

715

<b>NOTE:</b> There is one item in the list below that is not an actual string of digits... so it will not show up in the later code which identifies development names and assigns those names to BBLs

In [10]:
## NOTE: there's one BBL in here without any digits...
all_combined['BBL'].nunique()

451

## Extract development names and other information about the developments

In [11]:
rows = [] ## <-- empty list for later
current_dev_name = None 
current_dev_info = None

In [12]:
for page_num, page in enumerate(pdf.pages): # <-- for each page in the document...

    texts = page.extract_text() # <-- exract every table
    lines = texts.strip().split('\n')

    for line in lines:
        if len(line) < 36 and not line.startswith('Borough'):
            current_dev_name = line
        elif line.startswith('Borough'):
            current_dev_info = line
            line_parts = current_dev_info.split(':')
            boro = line_parts[1].strip(' TOTAL UNITS')
            units = line_parts[2].strip(' TDS #')
            transfer_date = line_parts[4]
        elif re.match(r'^\d{10}', line):
            parts = line.split()
        
            row = {
                    'dev_name': current_dev_name,
                    'bbl': parts[0],
                    'boro': boro,
                    'units': units,
                    'transfer': transfer_date
                }
            rows.append(row)

dev_df = pd.DataFrame(rows)

## Merge

In [13]:
## change to lowercase
all_combined.columns = all_combined.columns.str.lower()

In [14]:
## change names
all_combined.columns

Index(['bbl', 'bldg#', 'm', 'address', 'zip code', 'cd #', 'fc #', 'ss #',
       'sa #', 'cc #', 'bin#'],
      dtype='object')

In [15]:
## change column names
all_combined = all_combined.rename(columns = {'bldg#':'bldg',
                                              'sa #':'sa',
                                              'cc #': 'cc',
                                              'cd #':'cd',
                                              'fc #':'fc',
                                              'bin#':'bin',
                                              'ss #':'ss',
                                              'address':'og_address'})

In [16]:
## merge
merged_df = pd.merge(all_combined,
                     dev_df,
                     on = 'bbl',
                     how = 'left')

## FIX NULL DEVELOPMENTS

In [17]:
merged_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23069 entries, 0 to 23068
Data columns (total 15 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   bbl         23069 non-null  object
 1   bldg        23069 non-null  object
 2   m           23069 non-null  object
 3   og_address  23069 non-null  object
 4   zip code    23069 non-null  object
 5   cd          23069 non-null  object
 6   fc          23069 non-null  object
 7   ss          23069 non-null  object
 8   sa          23069 non-null  object
 9   cc          23069 non-null  object
 10  bin         23069 non-null  object
 11  dev_name    23045 non-null  object
 12  boro        23045 non-null  object
 13  units       23045 non-null  object
 14  transfer    23045 non-null  object
dtypes: object(15)
memory usage: 2.6+ MB


In [18]:
null_devs = merged_df[merged_df['dev_name'].isnull()].reset_index()
null_devs

,index,bbl,bldg,m,og_address,zip code,cd,fc,ss,sa,cc,bin,dev_name,boro,units,transfer
0,4136,,2,,1764 STERLING PLACE,11233,16,09,25,55,41,,NaN,NaN,NaN,NaN
1,4192,,2,,1776 STERLING PLACE,11233,16,09,25,55,41,,NaN,NaN,NaN,NaN
2,4217,,3,,510 HOWARD AVENUE,11233,16,09,25,55,41,,NaN,NaN,NaN,NaN
3,4540,,5,,553 RALPH AVENUE,11233,16,09,25,55,41,,NaN,NaN,NaN,NaN
4,4556,,5,,555 RALPH AVENUE,11233,16,09,25,55,41,,NaN,NaN,NaN,NaN
5,4915,,8,,1810 STERLING PLACE,11233,16,09,25,55,41,,NaN,NaN,NaN,NaN
6,4917,,8,,1812 STERLING PLACE,11233,16,09,25,55,41,,NaN,NaN,NaN,NaN
7,4918,,8,,1806 STERLING PLACE,11233,16,09,25,55,41,,NaN,NaN,NaN,NaN
8,12181,,1,M,344 EAST 28TH STREET,10016,06,12,59,74,02,,NaN,NaN,NaN,NaN
9,12198,,,,590 WARREN STREET,11217,06,10,26,52,39,,NaN,NaN,NaN,NaN


In [19]:
null_devs.og_address.unique().tolist()

['1764 STERLING PLACE',
 '1776 STERLING PLACE',
 '510 HOWARD AVENUE',
 '553 RALPH AVENUE',
 '555 RALPH AVENUE',
 '1810 STERLING PLACE',
 '1812 STERLING PLACE',
 '1806 STERLING PLACE',
 '344 EAST 28TH STREET',
 '590 WARREN STREET',
 '590 WARREN STREET -TEMPORARY ADDRESS',
 '65A SOUTH 10TH STREET',
 '1161 ADEE AVENUE',
 '1895 SCHIEFFELIN AVENUE',
 '1385 FRANKLIN AVENUE',
 'ADDRESS IS NOT AVAILABLE',
 '1695 SAINT JOHNS PLACE',
 '1697 SAINT JOHNS PLACE',
 '1701 SAINT JOHNS PLACE',
 '1758 STERLING PLACE',
 '514 HOWARD AVENUE',
 '',
 '188 TAPSCOTT STREET',
 'N/A OAKLAND PLACE']

In [20]:
dev_corrections = {'1764 STERLING PLACE':'HOWARD AVENUE-PARK PLACE',
 '1776 STERLING PLACE':'HOWARD AVENUE-PARK PLACE',
 '510 HOWARD AVENUE':'HOWARD AVENUE-PARK PLACE',
 '553 RALPH AVENUE':'HOWARD AVENUE-PARK PLACE',
 '555 RALPH AVENUE':'HOWARD AVENUE-PARK PLACE',
 '1810 STERLING PLACE':'HOWARD AVENUE-PARK PLACE',
 '1812 STERLING PLACE':'HOWARD AVENUE-PARK PLACE',
 '1806 STERLING PLACE':'HOWARD AVENUE-PARK PLACE',
 '344 EAST 28TH STREET':'344 EAST 28TH STREET',
 '590 WARREN STREET':'572 WARREN STREET',
 '590 WARREN STREET -TEMPORARY ADDRESS':'572 WARREN STREET',
 '65A SOUTH 10TH STREET':'BERRY STREET-SOUTH 9TH STREET',
 '1161 ADEE AVENUE':'EASTCHESTER GARDENS',
 '1895 SCHIEFFELIN AVENUE':'EDENWALD',
 '1385 FRANKLIN AVENUE':'FRANKLIN AVENUE II CONVENTIONAL',
 'ADDRESS IS NOT AVAILABLE':'HOPE GARDENS',
 '1695 SAINT JOHNS PLACE':'HOWARD AVENUE-PARK PLACE',
 '1697 SAINT JOHNS PLACE':'HOWARD AVENUE-PARK PLACE',
 '1701 SAINT JOHNS PLACE':'HOWARD AVENUE-PARK PLACE',
 '1758 STERLING PLACE':'HOWARD AVENUE-PARK PLACE',
 '514 HOWARD AVENUE':'HOWARD AVENUE-PARK PLACE',
 '':'HOWARD AVENUE-PARK PLACE',
 '188 TAPSCOTT STREET':'TAPSCOTT STREET REHAB',
 'N/A OAKLAND PLACE':'TWIN PARKS EAST (SITE 9)',
 ## the ones below are fixing some kind of dev mixup that happened along the way
 '699 EAST 139TH STREET':'BETANCES III, 13',
 '695 EAST 139TH STREET':'BETANCES III, 13',
 '3340 BAILEY AVENUE':'FORT INDEPENDENCE STREET-HEATH AVENUE',
 '3350 BAILEY AVENUE':'FORT INDEPENDENCE STREET-HEATH AVENUE',
 '3353 FORT INDEPENDENCE STREET':'FORT INDEPENDENCE STREET-HEATH AVENUE',
 '3355 FORT INDEPENDENCE STREET':'FORT INDEPENDENCE STREET-HEATH AVENUE',
 '120 EAST 123RD STREET':'PARK AVENUE-EAST 122ND, 123RD STREETS',
 '115 EAST 122ND STREET':'PARK AVENUE-EAST 122ND, 123RD STREETS',
 '114 EAST 123RD STREET':'PARK AVENUE-EAST 122ND, 123RD STREETS',
 '116 EAST 123RD STREET':'PARK AVENUE-EAST 122ND, 123RD STREETS',
 '1487 SAINT JOHNS PLACE':'STERLING PLACE REHABS (SAINT JOHNS-STERLING)',
'1483 SAINT JOHNS PLACE':'STERLING PLACE REHABS (SAINT JOHNS-STERLING)',
'1491 SAINT JOHNS PLACE':'STERLING PLACE REHABS (SAINT JOHNS-STERLING)',
'1506 STERLING PLACE':'STERLING PLACE REHABS (SAINT JOHNS-STERLING)',
'1511 STERLING PLACE':'STERLING PLACE REHABS (SAINT JOHNS-STERLING)',
'1640 STERLING PLACE':'STERLING PLACE REHABS (SAINT JOHNS-STERLING)',
'1448 STERLING PLACE':'STERLING PLACE REHABS (STERLING-BUFFALO)',
'1452 STERLING PLACE':'STERLING PLACE REHABS (STERLING-BUFFALO)',
'1568 STERLING PLACE':'STERLING PLACE REHABS (STERLING-BUFFALO)',
'1578 STERLING PLACE':'STERLING PLACE REHABS (STERLING-BUFFALO)',
'1588 STERLING PLACE':'STERLING PLACE REHABS (STERLING-BUFFALO)',
'1598 STERLING PLACE':'STERLING PLACE REHABS (STERLING-BUFFALO)',
'225 BUFFALO AVENUE':'STERLING PLACE REHABS (STERLING-BUFFALO)',
'450 WEST 164TH STREET':'WASHINGTON HEIGHTS REHAB PHASE III (FORT WASHINGTON)',
'457 WEST 164TH STREET':'WASHINGTON HEIGHTS REHAB PHASE III (FORT WASHINGTON)',
'461 WEST 164TH STREET':'WASHINGTON HEIGHTS REHAB PHASE III (FORT WASHINGTON)',
'463 WEST 164TH STREET':'WASHINGTON HEIGHTS REHAB PHASE III (FORT WASHINGTON)',
'465 WEST 164TH STREET':'WASHINGTON HEIGHTS REHAB PHASE III (FORT WASHINGTON)',
'500 WEST 164TH STREET':'WASHINGTON HEIGHTS REHAB PHASE III (FORT WASHINGTON)',
'2111 AMSTERDAM AVENUE':'WASHINGTON HEIGHTS REHAB PHASE III (FORT WASHINGTON)',
'2109 AMSTERDAM AVENUE':'WASHINGTON HEIGHTS REHAB PHASE III (FORT WASHINGTON)',
'545 WEST 156TH STREET':'WASHINGTON HEIGHTS REHAB PHASE III (HARLEM RIVER)',
'502 WEST 177TH STREET':'WASHINGTON HEIGHTS REHAB PHASE IV (C)',
'506 WEST 176TH STREET':'WASHINGTON HEIGHTS REHAB PHASE IV (C)',
'510 WEST 176TH STREET':'WASHINGTON HEIGHTS REHAB PHASE IV (D)',
'514 WEST 176TH STREET':'WASHINGTON HEIGHTS REHAB PHASE IV (D)',
'157 ALASKA STREET':'WEST BRIGHTON II',
'155 ALASKA STREET':'WEST BRIGHTON II',
'1115 CASTLETON AVENUE':'WEST BRIGHTON II',
'1085 CASTLETON AVENUE':'WEST BRIGHTON II',
'1083 CASTLETON AVENUE':'WEST BRIGHTON II',
'1065 CASTLETON AVENUE':'WEST BRIGHTON II',
'260 BROADWAY':'WEST BRIGHTON II',
'244 BROADWAY':'WEST BRIGHTON II',
'159 ALASKA STREET':'WEST BRIGHTON II',
'501 WEST 176TH STREET':'WASHINGTON HEIGHTS REHAB (GROUPS 1&2)',
'2340 AMSTERDAM AVENUE':'WASHINGTON HEIGHTS REHAB (GROUPS 1&2)',
'2346 AMSTERDAM AVENUE':'WASHINGTON HEIGHTS REHAB (GROUPS 1&2)',
'500 WEST 177TH STREET':'WASHINGTON HEIGHTS REHAB (GROUPS 1&2)',
'503 WEST 177TH STREET':'WASHINGTON HEIGHTS REHAB (GROUPS 1&2)',
'511 WEST 177TH STREET':'WASHINGTON HEIGHTS REHAB (GROUPS 1&2)',
'514 WEST 177TH STREET':'WASHINGTON HEIGHTS REHAB (GROUPS 1&2)',
'506 WEST 177TH STREET':'WASHINGTON HEIGHTS REHAB (GROUPS 1&2)',
'509 WEST 176TH STREET':'WASHINGTON HEIGHTS REHAB (GROUPS 1&2)',}

In [21]:
for addresses,dev_names in dev_corrections.items():
    merged_df.loc[merged_df['og_address'] == addresses, 'dev_name'] = dev_names

In [22]:
## but the right number of addresses and bins
print(f'number of unique address: {merged_df.og_address.nunique()} \nnumber of unique bins: {merged_df.bin.nunique()}')

number of unique address: 1538 
number of unique bins: 715


In [23]:
merged_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23069 entries, 0 to 23068
Data columns (total 15 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   bbl         23069 non-null  object
 1   bldg        23069 non-null  object
 2   m           23069 non-null  object
 3   og_address  23069 non-null  object
 4   zip code    23069 non-null  object
 5   cd          23069 non-null  object
 6   fc          23069 non-null  object
 7   ss          23069 non-null  object
 8   sa          23069 non-null  object
 9   cc          23069 non-null  object
 10  bin         23069 non-null  object
 11  dev_name    23069 non-null  object
 12  boro        23045 non-null  object
 13  units       23045 non-null  object
 14  transfer    23045 non-null  object
dtypes: object(15)
memory usage: 2.6+ MB


## CLEANING

In [24]:
## take a look at development names first
merged_df.dev_name.unique()

array(['ARMSTRONG I', 'ARMSTRONG II', '65A SOUTH 10TH STREET',
       'BETANCES III, 13', 'BETANCES IV', 'BETANCES V', 'BOULEVARD',
       'BUSHWICK II (GROUPS A & C)', 'BUSHWICK II CDA (GROUP E)',
       'CROWN HEIGHTS', 'EASTCHESTER GARDENS', 'EDENWALD',
       'FENIMORE-LEFFERTS', 'FRANKLIN AVENUE II CONVENTIONAL',
       'HARLEM RIVER', 'HOWARD AVENUE', 'HOWARD AVENUE-PARK PLACE',
       'LINDEN', 'OCEAN BAY APARTMENTS (BAYSIDE)', 'PARK ROCK REHAB',
       'SAMUEL (CITY)', 'SOUTH BRONX AREA (SITE 402)',
       'TWIN PARKS WEST (SITES 1 & 2)', 'WEEKSVILLE GARDENS',
       'WILLIAMS PLAZA', 'WILLIAMSBURG', '1010 EAST 178TH STREET',
       '104-14 TAPSCOTT STREET', '335 EAST 111TH STREET',
       '344 EAST 28TH STREET', '572 WARREN STREET', 'AUDUBON',
       'BAILEY AVENUE-WEST 193RD STREET', 'BAYCHESTER',
       'BELMONT-SUTTER AREA', 'BERRY STREET-SOUTH 9TH STREET',
       'BETANCES I', 'BETANCES II, 13', 'BETANCES II, 18',
       'BETANCES III, 18', 'BETANCES II, 9A', 'BETANCES III

In [25]:
test = merged_df[merged_df['dev_name'] == 'TWIN PARKS WEST (SITES 1 & 2)']
test.head()

,bbl,bldg,m,og_address,zip code,cd,fc,ss,sa,cc,bin,dev_name,boro,units,transfer
5696,2031430155,1,,360 FORD STREET,10457,05,15,33,86,15,2092394,TWIN PARKS WEST (SITES 1 & 2),BRONX,312,10/31/2018
5697,2031430155,1,,360 FORD STREET,10457,05,15,33,86,15,2092394,TWIN PARKS WEST (SITES 1 & 2),BRONX,312,10/31/2018
5698,2031430155,1,,360 FORD STREET,10457,05,15,33,86,15,2092394,TWIN PARKS WEST (SITES 1 & 2),BRONX,312,10/31/2018
5699,2031430155,1,,360 FORD STREET,10457,05,15,33,86,15,2092394,TWIN PARKS WEST (SITES 1 & 2),BRONX,312,10/31/2018
5700,2031430155,1,,360 FORD STREET,10457,05,15,33,86,15,2092394,TWIN PARKS WEST (SITES 1 & 2),BRONX,312,10/31/2018


In [26]:
## dict of corrections
dev_corrections = {"65A SOUTH 10TH STREET":"BERRY STREET-SOUTH 9TH STREET",
                   "N/A OAKLAND PLACE":"TWIN PARKS EAST (SITE 9)"}

In [27]:
## apply the corrections
merged_df['dev_name'] = merged_df['dev_name'].replace(dev_corrections)
merged_df = merged_df.reset_index(drop = True)

In [28]:
merged_df.boro.unique()

array(['BROOKLY', 'BRONX', 'MANH', nan, 'QUEE', 'EN ISLAND'], dtype=object)

In [29]:
## now move onto boros
boro_corrections = {'BROOKLY':'brooklyn',
                    'BRONX':'bronx',
                    'MANH':'manhattan',
                    'QUEE':'queens',
                    'EN ISLAND':'staten island'}

In [30]:
## apply the corrections
merged_df['boro'] = merged_df['boro'].replace(boro_corrections)
merged_df = merged_df.reset_index(drop = True)

In [31]:
## now addresses
merged_df.og_address.unique().tolist()

['346 CLIFTON PLACE',
 '344 CLIFTON PLACE',
 '360 NOSTRAND AVENUE',
 '260 LEXINGTON AVENUE',
 '262 LEXINGTON AVENUE',
 '264 LEXINGTON AVENUE',
 '254 LEXINGTON AVENUE',
 '252 LEXINGTON AVENUE',
 '252A LEXINGTON AVENUE',
 '250 LEXINGTON AVENUE',
 '366 CLIFTON PLACE',
 '380 CLIFTON PLACE',
 '388 CLIFTON PLACE',
 '396 CLIFTON PLACE',
 '545 GREENE AVENUE',
 '555 GREENE AVENUE',
 '304 LEXINGTON AVENUE',
 '302 LEXINGTON AVENUE',
 '300 LEXINGTON AVENUE',
 '298 LEXINGTON AVENUE',
 '296 LEXINGTON AVENUE',
 '294 LEXINGTON AVENUE',
 '292 LEXINGTON AVENUE',
 '290 LEXINGTON AVENUE',
 '288 LEXINGTON AVENUE',
 '286 LEXINGTON AVENUE',
 '284 LEXINGTON AVENUE',
 '282 LEXINGTON AVENUE',
 '280 LEXINGTON AVENUE',
 '278 LEXINGTON AVENUE',
 '310 LEXINGTON AVENUE',
 '320 LEXINGTON AVENUE',
 '330 LEXINGTON AVENUE',
 '336 LEXINGTON AVENUE',
 '435 GATES AVENUE',
 '441 GATES AVENUE',
 '447 GATES AVENUE',
 '449 GATES AVENUE',
 '451 GATES AVENUE',
 '453 GATES AVENUE',
 '455 GATES AVENUE',
 '457 GATES AVENUE',
 '459 

In [32]:
merged_df['new_address'] = merged_df['og_address']

In [33]:
address_corrections = {'590 WARREN STREET -TEMPORARY ADDRESS':'590 WARREN STREET',
                       '1149GAR 229TH DRIVE NORTH':'1149 229TH DRIVE NORTH',
                       '2185GAR REEDS MILL LANE':'2185 REEDS MILL LANE',
                       '778GAR HENDERSON AVENUE':'778 HENDERSON AVENUE',
                       '344GAR EAST 28TH STREET':'344 EAST 28TH STREET',
                       'A C POWELL BOULEVARD':'ADAM C POWELL BOULEVARD',
                       '48REAR GRAFTON STREET':'48 GRAFTON STREET',
                       '365 FORD STREET, WING A':'365 FORD STREET',
                       '365 FORD STREET, WING B':'365 FORD STREET',
                       '365 FORD STREET, WING C':'365 FORD STREET',
                       '365 FORD STREET, WING D':'365 FORD STREET',
                       '365 EAST 183RD STREET, WING D':'365 EAST 183RD STREET',
                       '365 EAST 183RD STREET, WING E':'365 EAST 183RD STREET',
                       '365 EAST 183RD STREET, WING F':'365 EAST 183RD STREET'}

In [34]:
## iterate through and make replacements
for old_add, new_add in address_corrections.items():
    merged_df['new_address'] = merged_df['new_address'].str.replace(old_add,new_add)

In [35]:
# cleaning up the names of the street addresses so that they match the HPD data
merged_df['new_address'] = merged_df['new_address'].str.replace(r'(\d+)(ST|ND|TH|RD)\b', r'\1', regex=True)

In [36]:
merged_df.dev_name.nunique()

101

In [37]:
merged_df.dev_name.unique()

array(['ARMSTRONG I', 'ARMSTRONG II', 'BERRY STREET-SOUTH 9TH STREET',
       'BETANCES III, 13', 'BETANCES IV', 'BETANCES V', 'BOULEVARD',
       'BUSHWICK II (GROUPS A & C)', 'BUSHWICK II CDA (GROUP E)',
       'CROWN HEIGHTS', 'EASTCHESTER GARDENS', 'EDENWALD',
       'FENIMORE-LEFFERTS', 'FRANKLIN AVENUE II CONVENTIONAL',
       'HARLEM RIVER', 'HOWARD AVENUE', 'HOWARD AVENUE-PARK PLACE',
       'LINDEN', 'OCEAN BAY APARTMENTS (BAYSIDE)', 'PARK ROCK REHAB',
       'SAMUEL (CITY)', 'SOUTH BRONX AREA (SITE 402)',
       'TWIN PARKS WEST (SITES 1 & 2)', 'WEEKSVILLE GARDENS',
       'WILLIAMS PLAZA', 'WILLIAMSBURG', '1010 EAST 178TH STREET',
       '104-14 TAPSCOTT STREET', '335 EAST 111TH STREET',
       '344 EAST 28TH STREET', '572 WARREN STREET', 'AUDUBON',
       'BAILEY AVENUE-WEST 193RD STREET', 'BAYCHESTER',
       'BELMONT-SUTTER AREA', 'BETANCES I', 'BETANCES II, 13',
       'BETANCES II, 18', 'BETANCES III, 18', 'BETANCES II, 9A',
       'BETANCES III, 9A', 'BETANCES VI', 'BE

In [38]:
print(f'number of unique addresses from old column:{merged_df.og_address.nunique()}\nnumber of unique address from new column:{merged_df.new_address.nunique()}')

number of unique addresses from old column:1538
number of unique address from new column:1526


In [39]:
## drop
dropped_df = merged_df.drop_duplicates(subset=['new_address','bin','bbl'],keep='first')

In [40]:
## check again
print(f'number of unique address: {dropped_df.new_address.nunique()} \nnumber of unique bins: {dropped_df.bin.nunique()}\nnumber of unique developments:{dropped_df.dev_name.nunique()}')

number of unique address: 1526 
number of unique bins: 715
number of unique developments:101


In [41]:
dropped_df.dev_name.unique().tolist()

['ARMSTRONG I',
 'ARMSTRONG II',
 'BERRY STREET-SOUTH 9TH STREET',
 'BETANCES III, 13',
 'BETANCES IV',
 'BOULEVARD',
 'BUSHWICK II (GROUPS A & C)',
 'BUSHWICK II CDA (GROUP E)',
 'CROWN HEIGHTS',
 'EASTCHESTER GARDENS',
 'EDENWALD',
 'FENIMORE-LEFFERTS',
 'FRANKLIN AVENUE II CONVENTIONAL',
 'HARLEM RIVER',
 'HOWARD AVENUE',
 'HOWARD AVENUE-PARK PLACE',
 'LINDEN',
 'OCEAN BAY APARTMENTS (BAYSIDE)',
 'PARK ROCK REHAB',
 'SAMUEL (CITY)',
 'SOUTH BRONX AREA (SITE 402)',
 'TWIN PARKS WEST (SITES 1 & 2)',
 'WEEKSVILLE GARDENS',
 'WILLIAMS PLAZA',
 'WILLIAMSBURG',
 '1010 EAST 178TH STREET',
 '104-14 TAPSCOTT STREET',
 '335 EAST 111TH STREET',
 '344 EAST 28TH STREET',
 '572 WARREN STREET',
 'AUDUBON',
 'BAILEY AVENUE-WEST 193RD STREET',
 'BAYCHESTER',
 'BELMONT-SUTTER AREA',
 'BETANCES I',
 'BETANCES II, 13',
 'BETANCES II, 18',
 'BETANCES II, 9A',
 'BETANCES III, 18',
 'BETANCES III, 9A',
 'BETANCES V',
 'BETANCES VI',
 'BETHUNE GARDENS',
 'BOSTON ROAD PLAZA',
 'BOSTON SECOR',
 'BUSHWICK II 

In [42]:
test = dropped_df[dropped_df['dev_name'] == 'WASHINGTON HEIGHTS REHAB (GROUPS 1&2)']
test.head()

,bbl,bldg,m,og_address,zip code,cd,fc,ss,sa,cc,bin,dev_name,boro,units,transfer,new_address
19976,1021320047,1,M,501 WEST 176TH STREET,10033,12,13,31,72,10,1063193,WASHINGTON HEIGHTS REHAB (GROUPS 1&2),manhattan,216,11/30/2020,501 WEST 176 STREET
19980,1021320047,1,,2340 AMSTERDAM AVENUE,10033,12,13,31,72,10,1063193,WASHINGTON HEIGHTS REHAB (GROUPS 1&2),manhattan,216,11/30/2020,2340 AMSTERDAM AVENUE
19984,1021320047,1,,2346 AMSTERDAM AVENUE,10033,12,13,31,72,10,1063193,WASHINGTON HEIGHTS REHAB (GROUPS 1&2),manhattan,216,11/30/2020,2346 AMSTERDAM AVENUE
19988,1021320047,1,M,500 WEST 177TH STREET,10033,12,13,31,72,10,1063193,WASHINGTON HEIGHTS REHAB (GROUPS 1&2),manhattan,216,11/30/2020,500 WEST 177 STREET
19992,1021320110,2,M,503 WEST 177TH STREET,10033,12,13,31,72,10,1063213,WASHINGTON HEIGHTS REHAB (GROUPS 1&2),manhattan,216,11/30/2020,503 WEST 177 STREET


In [43]:
dropped_df.dev_name.unique().tolist()

['ARMSTRONG I',
 'ARMSTRONG II',
 'BERRY STREET-SOUTH 9TH STREET',
 'BETANCES III, 13',
 'BETANCES IV',
 'BOULEVARD',
 'BUSHWICK II (GROUPS A & C)',
 'BUSHWICK II CDA (GROUP E)',
 'CROWN HEIGHTS',
 'EASTCHESTER GARDENS',
 'EDENWALD',
 'FENIMORE-LEFFERTS',
 'FRANKLIN AVENUE II CONVENTIONAL',
 'HARLEM RIVER',
 'HOWARD AVENUE',
 'HOWARD AVENUE-PARK PLACE',
 'LINDEN',
 'OCEAN BAY APARTMENTS (BAYSIDE)',
 'PARK ROCK REHAB',
 'SAMUEL (CITY)',
 'SOUTH BRONX AREA (SITE 402)',
 'TWIN PARKS WEST (SITES 1 & 2)',
 'WEEKSVILLE GARDENS',
 'WILLIAMS PLAZA',
 'WILLIAMSBURG',
 '1010 EAST 178TH STREET',
 '104-14 TAPSCOTT STREET',
 '335 EAST 111TH STREET',
 '344 EAST 28TH STREET',
 '572 WARREN STREET',
 'AUDUBON',
 'BAILEY AVENUE-WEST 193RD STREET',
 'BAYCHESTER',
 'BELMONT-SUTTER AREA',
 'BETANCES I',
 'BETANCES II, 13',
 'BETANCES II, 18',
 'BETANCES II, 9A',
 'BETANCES III, 18',
 'BETANCES III, 9A',
 'BETANCES V',
 'BETANCES VI',
 'BETHUNE GARDENS',
 'BOSTON ROAD PLAZA',
 'BOSTON SECOR',
 'BUSHWICK II 

In [44]:
dropped_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1531 entries, 0 to 23061
Data columns (total 16 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   bbl          1531 non-null   object
 1   bldg         1531 non-null   object
 2   m            1531 non-null   object
 3   og_address   1531 non-null   object
 4   zip code     1531 non-null   object
 5   cd           1531 non-null   object
 6   fc           1531 non-null   object
 7   ss           1531 non-null   object
 8   sa           1531 non-null   object
 9   cc           1531 non-null   object
 10  bin          1531 non-null   object
 11  dev_name     1531 non-null   object
 12  boro         1508 non-null   object
 13  units        1508 non-null   object
 14  transfer     1508 non-null   object
 15  new_address  1531 non-null   object
dtypes: object(16)
memory usage: 203.3+ KB


In [45]:
## write to a csv
dropped_df.to_csv('../input/coded_files/addresses_updated_120525.csv',index=False)